# Regression

Regression predicts a number. This notebook builds from the simplest possible model — one feature
and a straight line — to a multiple regression and a polynomial, using the red wine chemistry data
to predict sensory quality.

## Learning objectives

By the end of this notebook you will be able to:

- fit a simple linear regression and interpret its slope and intercept;
- extend to multiple regression and read standardised coefficients;
- add polynomial terms and judge whether they help on held-out data;
- report MAE, RMSE, and R² and compare against a baseline;
- explain what the residuals reveal about the model's assumptions.

## Concept

**Simple linear regression** fits `y = b + w x` by least squares. The slope `w` is the change in the
target for a one-unit change in the feature; the intercept `b` is the prediction when the feature
is zero (often not physically meaningful).

**Multiple regression** extends this to several features. When features are on different scales,
standardising them makes the coefficients comparable, so you can say which feature matters most.
Correlated features, however, split credit between themselves and make individual coefficients
unstable.

**Polynomial regression** adds powers and interactions, letting a linear model curve. More degrees
mean more flexibility and a higher risk of overfitting; the test set is the arbiter.

The wine `quality` score is an **ordinal** integer (roughly 3–8). Treating it as a continuous target
is a modelling convenience; the residuals will cluster on the integer values, which is a clue that a
different model might suit the question better.

## Worked example

### Load and inspect

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import ml
from ds_practice import load_wine, set_seed, regression_metrics

set_seed(42)
wine = load_wine()
print("shape:", wine.shape)
display(wine.head())
print("\ntarget counts:")
print(wine["quality"].value_counts().sort_index().to_string())

### Simple regression: quality from alcohol

Alcohol is the single strongest chemical predictor of the quality score. We fit it alone first as a
baseline.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = wine[["alcohol"]]
y = wine["quality"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

simple = LinearRegression().fit(X_train, y_train)
print("slope:", round(simple.coef_[0], 3), "| intercept:", round(simple.intercept_, 3))
print("test metrics:", {k: round(v, 3) for k, v in regression_metrics(y_test, simple.predict(X_test)).items()})

### Multiple regression

Using every chemical feature usually improves on alcohol alone. Scaling keeps the coefficients on a
common footing.

In [ ]:
features = [c for c in wine.columns if c != "quality"]
Xm = wine[features]
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, y, test_size=0.2, random_state=42)

multiple = ml.make_regressor("linear").fit(Xm_train, ym_train)
print("test metrics:", {k: round(v, 3) for k, v in regression_metrics(ym_test, multiple.predict(Xm_test)).items()})

coefs = pd.Series(multiple.named_steps["model"].coef_, index=features).sort_values(key=abs, ascending=False)
print("\nlargest standardised coefficients:")
display(coefs.head(6).round(3))

### Polynomial terms

A quadratic term on alcohol lets the relationship bend. We compare degrees one and two on the same
split.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rows = []
for degree in (1, 2):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("model", LinearRegression()),
    ]).fit(Xm_train, ym_train)
    rows.append({"degree": degree, **{k: round(v, 3) for k, v in regression_metrics(ym_test, model.predict(Xm_test)).items()}})
display(pd.DataFrame(rows))

### Residuals

Residuals clustered on integer values reveal that the target is not truly continuous; the model
cannot predict 5.5.

In [ ]:
pred = multiple.predict(Xm_test)
residuals = pd.Series(ym_test - pred)
print(residuals.describe().round(3).to_string())
print("\nunique residual values (rounded):", sorted(np.round(residuals.unique(), 2))[:10])

## Exercises

1. **One feature, three metrics.** Fit simple regressions of `quality` on each of `alcohol`,
   `volatile acidity`, and `citric acid`. Rank them by test RMSE and by the sign of the slope.
2. **Interactions.** Add `alcohol * volatile acidity` as a feature and report whether test R²
   improves. Explain why an interaction can capture a joint effect.
3. **Ordinal reality.** Discretise the predictions by rounding to the nearest integer and report
   the share that exactly match the true `quality`. Why is this a fairer measure of usefulness?

## Limitations

Treating an ordinal score as continuous ignores the unequal spacing between categories; an ordinal
regression or a classifier would be more faithful. Wine samples are not a random sample of all
wine, so the model describes this dataset only. R² is modest because chemistry explains part, not
all, of a subjective sensory rating. Finally, the linear model assumes constant error variance,
which the integer residuals violate.